# 🚀 کاربرد پیشرفته Climatology Engine

این نوت‌بوک کاربردهای پیشرفته موتور اقلیم‌شناسی را نشان می‌دهد.

**مواردی که یاد می‌گیرید:**
- کار با داده‌های شبکه‌ای (Gridded Data)
- حالت حدی (Extreme Value Mode) با GEV
- تنظیمات پیشرفته `config.yaml`
- استفاده از checkpoint برای ادامه پردازش
- مدیریت حافظه و block_size
- ذخیره‌سازی در فرمت‌های مختلف (Zarr, NetCDF, CSV)
- تحلیل خروجی با xarray
- رسم نقشه‌های پیشرفته با Cartopy
- بهینه‌سازی عملکرد برای داده‌های بزرگ

---

## 📐 تنظیمات پیشرفته config.yaml

فایل `config.yaml` تمام پارامترهای پردازش را کنترل می‌کند.

### پارامترهای کلیدی

| پارامتر | توضیح | مقدار پیش‌فرض |
|---------|-------|---------------|
| `block_size` | تعداد نقاط در هر بلوک | ۱۰۰۰ |
| `n_points_max` | حداکثر تعداد نقاط | ۴۰۰۰۰ |
| `use_extreme_values` | فعال‌سازی حالت حدی | false |
| `data_format` | فرمت داده (station/gridded) | auto |
| `compression` | الگوریتم فشرده‌سازی | zstd |
| `cache_enabled` | فعال‌سازی کش دیسک | true |

### تنظیمات حالت حدی

```yaml
window:
  days: 2
  use_extreme_values: true   # فعال‌سازی GEV
```

### تنظیمات داده شبکه‌ای

```yaml
data_format: "gridded"
lat_min: 25.0
lat_max: 40.0
lon_min: 44.0
lon_max: 64.0
```

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# ============================================================================
# ۱. بارگذاری و بررسی config.yaml
# ============================================================================

config_path = os.path.join(project_root, 'config.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    print("📋 تنظیمات فعلی:")
    print("=" * 60)
    
    # نمایش پارامترهای کلیدی
    print(f"block_size: {config.get('processing', {}).get('block_size', 'N/A')}")
    print(f"n_points_max: {config.get('processing', {}).get('n_points_max', 'N/A')}")
    print(f"use_extreme_values: {config.get('window', {}).get('use_extreme_values', 'N/A')}")
    print(f"data_format: {config.get('data_format', 'N/A')}")
    print(f"compression: {config.get('processing', {}).get('compression', 'N/A')}")
    print("=" * 60)
else:
    print("⚠️ فایل config.yaml یافت نشد.")

In [ ]:
# ============================================================================
# ۲. تولید داده شبکه‌ای مصنوعی (Gridded Data)
# ============================================================================

def create_gridded_data(n_lat=20, n_lon=30, n_time=365):
    """
    تولید داده شبکه‌ای مصنوعی با ابعاد (time, lat, lon)
    """
    np.random.seed(42)
    lat = np.linspace(25, 40, n_lat)
    lon = np.linspace(44, 64, n_lon)
    
    # تولید داده با الگوی سینوسی + نویز
    time = np.arange(n_time)
    data = np.zeros((n_time, n_lat, n_lon))
    
    for i, t in enumerate(time):
        seasonal = 15 + 10 * np.sin(2 * np.pi * t / 365)
        spatial = np.outer(np.sin(lat * 0.1), np.cos(lon * 0.05))
        noise = np.random.normal(0, 2, (n_lat, n_lon))
        data[i, :, :] = seasonal + spatial * 5 + noise
    
    return lat, lon, data

lat, lon, gridded_data = create_gridded_data(n_lat=20, n_lon=30, n_time=365)

print("📊 داده شبکه‌ای مصنوعی:")
print(f"   ابعاد: {gridded_data.shape}")
print(f"   طول‌ها: {len(lat)} نقطه (از {lat.min():.1f} تا {lat.max():.1f})")
print(f"   عرض‌ها: {len(lon)} نقطه (از {lon.min():.1f} تا {lon.max():.1f})")
print(f"   بازه داده: [{gridded_data.min():.2f}, {gridded_data.max():.2f}]")

In [ ]:
# ذخیره داده شبکه‌ای در فایل NetCDF
netcdf_path = os.path.join(project_root, 'sample_data', 'gridded_sample.nc')
os.makedirs(os.path.dirname(netcdf_path), exist_ok=True)

ds_netcdf = xr.Dataset(
    data_vars={
        'tmean': (('time', 'lat', 'lon'), gridded_data),
    },
    coords={
        'time': np.arange(365),
        'lat': lat,
        'lon': lon,
    },
    attrs={'description': 'Synthetic gridded temperature data'}
)

ds_netcdf.to_netcdf(netcdf_path)
print(f"✅ داده شبکه‌ای در {netcdf_path} ذخیره شد.")

In [ ]:
# بارگذاری داده شبکه‌ای از NetCDF
ds_loaded = xr.open_dataset(netcdf_path)
print("📂 داده شبکه‌ای بارگذاری شد:")
print(f"   ابعاد: {ds_loaded.dims}")
print(f"   متغیرها: {list(ds_loaded.data_vars)}")
print(f"   مختصات: {list(ds_loaded.coords)}")

In [ ]:
# ============================================================================
# ۳. برازش روی داده شبکه‌ای (تبدیل به سری زمانی)
# ============================================================================

from core.engine.plugin_loader import load_plugins
from core.engine.distribution_plugin import DistributionPlugin

# بارگذاری پلاگین‌ها
plugins = load_plugins()
distributions = {dist.name: dist for dist in plugins.values()}

# انتخاب یک نقطه از شبکه
lat_idx = 10
lon_idx = 15
point_data = gridded_data[:, lat_idx, lon_idx]

print(f"📊 داده نقطه ({lat[lat_idx]:.2f}°, {lon[lon_idx]:.2f}°):")
print(f"   میانگین: {np.mean(point_data):.2f}°C")
print(f"   انحراف معیار: {np.std(point_data):.2f}°C")

# برازش توزیع نرمال روی نقطه
normal_dist = distributions['Normal']
result = normal_dist.fit(point_data)

print("\n📈 نتایج برازش:")
for key, value in result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# رسم سری زمانی نقطه انتخاب شده
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(point_data, color='blue', alpha=0.7, linewidth=1.5)
ax.axhline(np.mean(point_data), color='red', linestyle='--', linewidth=2, label=f'میانگین = {np.mean(point_data):.2f}°C')
ax.set_xlabel('زمان (روز)', fontsize=12)
ax.set_ylabel('دما (°C)', fontsize=12)
ax.set_title(f'سری زمانی در نقطه ({lat[lat_idx]:.2f}°, {lon[lon_idx]:.2f}°)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ۴. حالت حدی (Extreme Value Mode) با GEV
# ============================================================================

print("🔥 حالت حدی (Extreme Value Mode):")
print("=" * 60)

# استخراج بیشینه‌های فصلی
seasonal_max = []
season_names = ['زمستان', 'بهار', 'تابستان', 'پاییز']
season_idx = [(0, 90), (91, 181), (182, 273), (274, 364)]

for start, end in season_idx:
    seasonal_max.append(np.max(point_data[start:end]))

print(f"📊 بیشینه فصلی: {[f'{m:.2f}°C' for m in seasonal_max]}")

# برازش GEV روی بیشینه‌ها (اگر موجود باشد)
if 'GEV' in distributions:
    try:
        gev_dist = distributions['GEV']
        gev_result = gev_dist.fit(np.array(seasonal_max))
        print("\n📈 نتایج برازش GEV:")
        for key, value in gev_result.items():
            if isinstance(value, float):
                print(f"   {key}: {value:.4f}")
            else:
                print(f"   {key}: {value}")
    except Exception as e:
        print(f"⚠️ خطا در برازش GEV: {e}")
        print("   (GEV ممکن است به داده بیشتری نیاز داشته باشد)")
else:
    print("⚠️ توزیع GEV در دسترس نیست.")

In [ ]:
# ============================================================================
# ۵. ذخیره در فرمت‌های مختلف
# ============================================================================

output_dir = os.path.join(project_root, 'advanced_output')
os.makedirs(output_dir, exist_ok=True)

print("💾 ذخیره نتایج در فرمت‌های مختلف:")
print("=" * 60)

# ۵-۱. ذخیره در Zarr
zarr_path = os.path.join(output_dir, 'results.zarr')
ds_results = xr.Dataset(
    data_vars={
        'mean': (('lat', 'lon'), np.full((len(lat), len(lon)), np.mean(point_data))),
        'std': (('lat', 'lon'), np.full((len(lat), len(lon)), np.std(point_data))),
        'max': (('lat', 'lon'), np.full((len(lat), len(lon)), np.max(point_data))),
        'min': (('lat', 'lon'), np.full((len(lat), len(lon)), np.min(point_data))),
    },
    coords={'lat': lat, 'lon': lon}
)
ds_results.to_zarr(zarr_path, mode='w', consolidated=False)
print(f"✅ Zarr: {zarr_path}")

# ۵-۲. ذخیره در NetCDF
netcdf_out = os.path.join(output_dir, 'results.nc')
ds_results.to_netcdf(netcdf_out)
print(f"✅ NetCDF: {netcdf_out}")

# ۵-۳. ذخیره در CSV
csv_path = os.path.join(output_dir, 'results.csv')
df_csv = pd.DataFrame({
    'lat': lat,
    'lon': lon,
    'mean': np.full(len(lat), np.mean(point_data)),
    'std': np.full(len(lat), np.std(point_data)),
    'max': np.full(len(lat), np.max(point_data)),
    'min': np.full(len(lat), np.min(point_data))
})
df_csv.to_csv(csv_path, index=False)
print(f"✅ CSV: {csv_path}")

In [ ]:
# ============================================================================
# ۶. تحلیل خروجی Zarr با xarray
# ============================================================================

print("📊 تحلیل خروجی Zarr:")
print("=" * 60)

ds_zarr = xr.open_zarr(zarr_path, consolidated=False)
print(f"   ابعاد: {ds_zarr.dims}")
print(f"   متغیرها: {list(ds_zarr.data_vars)}")
print(f"   مختصات: {list(ds_zarr.coords)}")

# محاسبه میانگین مکانی
spatial_mean = ds_zarr['mean'].mean().values
print(f"   میانگین مکانی: {spatial_mean:.2f}°C")

ds_zarr.close()

In [ ]:
# ============================================================================
# ۷. رسم نقشه‌های پیشرفته
# ============================================================================

print("🗺️ رسم نقشه‌های پیشرفته:")

# ایجاد داده مصنوعی برای نقشه
map_data = np.zeros((len(lat), len(lon)))
for i in range(len(lat)):
    for j in range(len(lon)):
        map_data[i, j] = 15 + 5 * np.sin(lat[i] * 0.1) * np.cos(lon[j] * 0.05) + np.random.normal(0, 1)

fig, ax = plt.subplots(figsize=(12, 8))

im = ax.contourf(lon, lat, map_data, 20, cmap='RdBu_r')
cbar = plt.colorbar(im, ax=ax, label='دما (°C)')

ax.set_xlabel('طول جغرافیایی', fontsize=12)
ax.set_ylabel('عرض جغرافیایی', fontsize=12)
ax.set_title('نقشه دمای میانگین فضایی', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# رسم نقشه با Cartopy (اگر موجود باشد)
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')
    
    # رسم داده
    im = ax.contourf(lon, lat, map_data, 20, cmap='RdBu_r', 
                     transform=ccrs.PlateCarree())
    cbar = plt.colorbar(im, ax=ax, label='دما (°C)', shrink=0.7)
    
    ax.set_extent([44, 64, 25, 40], crs=ccrs.PlateCarree())
    ax.set_title('نقشه مکانی با Cartopy', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️ Cartopy در دسترس نیست.")

In [ ]:
# ============================================================================
# ۸. مدیریت حافظه و block_size
# ============================================================================

print("💾 مدیریت حافظه:")
print("=" * 60)

# شبیه‌سازی پردازش بلوکی
def process_in_blocks(data, block_size=100):
    """پردازش داده به صورت بلوکی برای کاهش مصرف حافظه"""
    n_points = data.shape[0]
    results = []
    
    for start in range(0, n_points, block_size):
        end = min(start + block_size, n_points)
        block = data[start:end]
        # پردازش بلوک
        block_result = {
            'start': start,
            'end': end,
            'mean': np.mean(block),
            'std': np.std(block),
            'size': len(block)
        }
        results.append(block_result)
        print(f"   بلوک {start//block_size + 1}: نقاط {start}-{end}, "
              f"میانگین = {block_result['mean']:.2f}")
    
    return results

# ایجاد داده بزرگ مصنوعی
large_data = np.random.randn(1000, 10)
print(f"📊 داده بزرگ: {large_data.shape}")

results = process_in_blocks(large_data, block_size=100)
print(f"\n✅ تعداد بلوک‌ها: {len(results)}")

In [ ]:
# ============================================================================
# ۹. استفاده از Checkpoint
# ============================================================================

from monitoring.checkpoint import save_checkpoint, load_checkpoint
import time

print("💾 سیستم Checkpoint:")
print("=" * 60)

checkpoint_dir = os.path.join(output_dir, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

# ذخیره checkpoint
save_checkpoint(checkpoint_dir, block=10, station=500)
print(f"✅ Checkpoint ذخیره شد: block=10, station=500")

# بارگذاری checkpoint
cp = load_checkpoint(checkpoint_dir)
if cp:
    print(f"📋 Checkpoint بارگذاری شد:")
    print(f"   block: {cp.get('block', 'N/A')}")
    print(f"   station: {cp.get('station', 'N/A')}")
    print(f"   timestamp: {cp.get('timestamp', 'N/A')}")
    print(f"   version: {cp.get('version', 'N/A')}")
else:
    print("⚠️ Checkpoint یافت نشد.")

In [ ]:
# ============================================================================
# ۱۰. بهینه‌سازی عملکرد
# ============================================================================

print("⚡ بهینه‌سازی عملکرد:")
print("=" * 60)

# مقایسه زمان اجرا با و بدون Numba
try:
    from numba import njit
    
    @njit
    def fast_calculation(data):
        return np.mean(data), np.std(data)
    
    data_test = np.random.randn(1000000)
    
    import time
    start = time.time()
    mean, std = fast_calculation(data_test)
    numba_time = time.time() - start
    print(f"✅ با Numba: {numba_time:.4f} ثانیه")
    
    start = time.time()
    mean, std = np.mean(data_test), np.std(data_test)
    numpy_time = time.time() - start
    print(f"   بدون Numba (NumPy): {numpy_time:.4f} ثانیه")
    print(f"   سرعت: {numpy_time/numba_time:.1f}x سریع‌تر")
    
except ImportError:
    print("⚠️ Numba نصب نیست. برای نصب: pip install numba")

In [ ]:
# ============================================================================
# ۱۱. خلاصه و نکات نهایی
# ============================================================================

print("📋 خلاصه کاربردهای پیشرفته:")
print("=" * 60)
print("""
✅ کار با داده‌های شبکه‌ای (Gridded Data)
✅ حالت حدی (Extreme Value Mode) با GEV
✅ تنظیمات پیشرفته config.yaml
✅ استفاده از checkpoint برای ادامه پردازش
✅ مدیریت حافظه و block_size
✅ ذخیره‌سازی در فرمت‌های مختلف (Zarr, NetCDF, CSV)
✅ تحلیل خروجی با xarray
✅ رسم نقشه‌های پیشرفته با Cartopy
✅ بهینه‌سازی عملکرد با Numba
""")

print("=" * 60)
print("🎉 کاربرد پیشرفته کامل شد!")
print(f"📁 خروجی‌ها در: {output_dir}")
print("=" * 60)